In [1]:
#task 1
#24k-0810
import heapq

def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def best_first_search(maze, start, goals):
    path = [start]
    current_pos = start
    remaining_goals = set(goals)

    while remaining_goals:
        next_goal = min(remaining_goals, key=lambda g: manhattan_distance(current_pos, g)) #closest goal
        segment = find_path(maze, current_pos, next_goal)
        if not segment:
            return None
        path.extend(segment[1:]) #current state updation
        current_pos = next_goal
        remaining_goals.remove(next_goal)
       
    return path

def find_path(maze, start, goal):
    """Standard Greedy Best-First Search for a single target"""
    pq = [(0, start, [start])]
    visited = {start}
   
    while pq:
        _, current, path = heapq.heappop(pq)
       
        if current == goal:
            return path
       
        for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            neighbor = (current[0] + dx, current[1] + dy)
           
            if (0 <= neighbor[0] < len(maze) and
                0 <= neighbor[1] < len(maze[0]) and
                maze[neighbor[0]][neighbor[1]] == 0 and
                neighbor not in visited):
               
                visited.add(neighbor)
                priority = manhattan_distance(neighbor, goal)
                heapq.heappush(pq, (priority, neighbor, path + [neighbor]))
    return None

maze = [
    [0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]
start_pos = (0, 0)
goals = [(0, 4), (4, 4), (4, 0)]

full_path = best_first_search(maze, start_pos, goals)
print(f"Path to visit all goals: {full_path}")

Path to visit all goals: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 1), (4, 0), (4, 1), (4, 2), (3, 2), (3, 3), (3, 4), (4, 4), (3, 4), (2, 4), (1, 4), (0, 4)]


In [4]:
#task 2
import heapq
import random

class DynamicAStar:
    def __init__(self, graph, heuristic):
        self.graph = graph
        self.heuristic = heuristic

    def a_star(self, start, goal):
        open_list = []
        heapq.heappush(open_list, (0, start))
        g = {start: 0}
        parent = {}

        while open_list:
            _, node = heapq.heappop(open_list)

            if node == goal:
                return self.reconstruct_path(parent, goal), g[goal]

            for neighbor in self.graph[node]:
                cost = self.graph[node][neighbor]
                new_g = g[node] + cost

                if neighbor not in g or new_g < g[neighbor]:
                    g[neighbor] = new_g
                    f = new_g + self.heuristic(neighbor, goal)
                    heapq.heappush(open_list, (f, neighbor))
                    parent[neighbor] = node

        return None, float('inf')

    def reconstruct_path(self, parent, node):
        path = [node]
        while node in parent:
            node = parent[node]
            path.append(node)
        return path[::-1]

    def update_edge(self, u, v, new_cost):
        self.graph[u][v] = new_cost


def heuristic(a, b):
    return abs(a - b)

def simulate_dynamic_changes(graph):
    u = random.choice(list(graph.keys()))
    if graph[u]:
        v = random.choice(list(graph[u].keys()))
        graph[u][v] = random.randint(1, 10)

graph = {
    0: {1: 2, 2: 4},
    1: {2: 1, 3: 7},
    2: {4: 3},
    3: {5: 1},
    4: {3: 2, 5: 5},
    5: {}
}
start = 0
goal = 5
dastar = DynamicAStar(graph, heuristic)
print("Initial Graph:")
for node in graph:
    print(node, "->", graph[node])

path, cost = dastar.a_star(start, goal) #intital a*
print("\nInitial Path:", path)
print("Initial Cost:", cost)

u = random.choice(list(graph.keys()))
if graph[u]:
    v = random.choice(list(graph[u].keys()))
    old_cost = graph[u][v]
    new_cost = random.randint(1, 10)

    print("\n--- Environment Change ---")
    print(f"Edge cost updated: {u} -> {v}")
    print(f"Old Cost: {old_cost}")
    print(f"New Cost: {new_cost}")

    dastar.update_edge(u, v, new_cost)


path, cost = dastar.a_star(start, goal) #a* after change
print("Updated Path:", path)
print("Updated Cost:", cost)

Initial Graph:
0 -> {1: 2, 2: 4}
1 -> {2: 1, 3: 7}
2 -> {4: 3}
3 -> {5: 1}
4 -> {3: 2, 5: 5}
5 -> {}

Initial Path: [0, 1, 2, 4, 3, 5]
Initial Cost: 9

--- Environment Change ---
Edge cost updated: 1 -> 2
Old Cost: 1
New Cost: 10
Updated Path: [0, 1, 3, 5]
Updated Cost: 10


In [10]:
#task 3
import heapq
import math

class DeliveryPoint:
    def __init__(self, name, x, y, start_time, end_time):
        self.name = name
        self.x = x
        self.y = y
        self.start_time = start_time
        self.end_time = end_time

def distance(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)

def greedy_delivery_route(start, deliveries):
    current = start
    current_time = 0
    route = []
    remaining = deliveries[:]

    while remaining:
        pq = []
        for point in remaining:
            travel_time = distance(current, point)
            arrival_time = current_time + travel_time

            if arrival_time <= point.end_time:
                urgency = point.end_time - arrival_time
                priority = urgency + travel_time
                heapq.heappush(pq, (priority, point, arrival_time))

        if not pq:
            print("No feasible delivery schedule!")
            return route

        _, next_point, arrival_time = heapq.heappop(pq)

        current_time = max(arrival_time, next_point.start_time)
        route.append(next_point.name)
        current = next_point
        remaining.remove(next_point)

    return route

start = DeliveryPoint("Warehouse", 0, 0, 0, float('inf'))
deliveries = [
    DeliveryPoint("A", 2, 3, 1, 10),
    DeliveryPoint("B", 5, 1, 2, 12),
    DeliveryPoint("C", 6, 4, 5, 15),
    DeliveryPoint("D", 1, 7, 5, 17)
]
print("Delivery Points and Time Windows:")
for d in deliveries:
    print(f"{d.name} -> Location({d.x},{d.y})  Time Window: {d.start_time}-{d.end_time}")

route = greedy_delivery_route(start, deliveries)

print("\nOptimized Delivery Route:")
print(" -> ".join(route))

Delivery Points and Time Windows:
A -> Location(2,3)  Time Window: 1-10
B -> Location(5,1)  Time Window: 2-12
C -> Location(6,4)  Time Window: 5-15
D -> Location(1,7)  Time Window: 5-17

Optimized Delivery Route:
A -> B -> C -> D
